In [2]:
import numpy
from alternet.annotation import *
from alternet.data_preprocessing import standardize_dataframe
import numpy
import os
import os.path as op






In [3]:


data_path = "/data/bionets/og86asub/alternet-project/alternet/data"
results_path = "/data/bionets/og86asub/alternet-project/alternet/results-2.0.1"

# Reference files
appris_path = "appris_data.appris.txt"
digger_path = "digger_data.csv"
biomart_path = "biomart.txt"
tf_list_path = "allTFs_hg38.txt"
sf_list_path = "splicefactors.csv"

# Expression data
gtex_transcript_tpm_path = "GTEx_Analysis_v10_RSEMv1.3.3_transcripts_tpm.txt"
gtex_sample_attributes_path = "GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt"

# Tissue to analyze
TISSUE = "Liver"
CONDITION = TISSUE

# Number of GRNBoost2 runs
N_RUNS = 1

os.makedirs(results_path, exist_ok=True)



In [4]:
biomart = pd.read_csv(op.join(data_path, biomart_path), sep='\t')
tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2tx = biomart.groupby('Gene stable ID')['Transcript stable ID'].apply(set).to_dict()
appris_df = pd.read_csv(op.join(data_path,appris_path), sep='\t')
digger_df = pd.read_csv(op.join(data_path,digger_path), low_memory=False)
# Load and map TF list
tf_list_raw = pd.read_csv(op.join(data_path,tf_list_path), sep='\t', header=None)
tf_list = map_tf_ids(tf_list_raw, biomart)

In [5]:


VARIANCE_PERCENTILE = 0.7  # Keep top 30%

In [6]:
# Load and map SF list
sf_list_raw = pd.read_csv(op.join(data_path, sf_list_path), header=0, sep = ',')
sf_list = map_sf_ids(sf_list_raw.loc[:, ['Splicing_Factor']], biomart)
# Combine TF and SF lists
regulator_list = combine_tf_sf_lists(tf_list, sf_list)
tx_to_regtype = dict(zip(regulator_list['Transcript stable ID'], regulator_list['Regulator_type']))
gene_to_regtype = regulator_list.groupby('Gene stable ID')['Regulator_type'].first().to_dict()


In [7]:


# Create mappings
transcript_mapper = create_transcript_mapping(biomart)
print(f"Transcript-to-gene mappings: {len(transcript_mapper)}")

# Annotation databases
tf_database = create_transcipt_annotation_database(
    tf_list=tf_list, appris_df=appris_df, digger=digger_df
)
regulator_database = create_transcipt_annotation_database(
    tf_list=regulator_list, appris_df=appris_df, digger=digger_df
)
print(f"TF annotation database: {len(tf_database)} entries")
print(f"Regulator annotation database: {len(regulator_database)} entries")



Transcript-to-gene mappings: 278220
TF annotation database: 16298 entries
Regulator annotation database: 18958 entries


In [8]:
from alternet.gtex_dataloader import *

In [9]:
gtex_data_dir = '/data/bionets/datasets/hackathon/data/GTEX'
params = {'sample_attributes': op.join(gtex_data_dir, 'GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt'), 'tissue': 'Liver', 'transcript_data':op.join(gtex_data_dir, 'GTEx_Analysis_2017-06-05_v8_RSEMv1.3.0_transcript_tpm.gct')}
tissue_ids = retrieve_GTEX_tissue_sampleids(params['sample_attributes'], tissue=params['tissue'])
transcript_data = read_GTEX_transcript_expression(params['transcript_data'], tissue_ids)
transcript_data = clean_GTEX_tissue_transcript_counts(transcript_data, biomart)
transcript_data = variance_filtering(transcript_data)


Retrieving tissue sample IDs
Reading Transcript expression data
Cleaning up counts


In [10]:

# sample_cols = [c for c in transcript_data.columns if c not in ['transcript_id', 'gene_id']]
# gene_data = transcript_data.groupby('gene_id')[sample_cols].sum().reset_index()



In [11]:

# # Create expression matrices (samples × features)
# gene_data = gene_data.set_index('gene_id')[sample_cols]
# transcript_data = transcript_data.set_index(['transcript_id', 'gene_id'])[sample_cols]



In [12]:
transcript_data

,transcript_id,gene_id,GTEX-11DXY-0526-SM-5EGGQ,GTEX-11DXZ-0126-SM-5EGGY,GTEX-11EQ9-0526-SM-5A5JZ,GTEX-11GSP-0626-SM-5986T,GTEX-11NUK-1226-SM-5P9GM,GTEX-11NV4-1326-SM-5HL6V,GTEX-11OF3-0726-SM-5BC4Z,GTEX-11TT1-1726-SM-5EQLJ,...,GTEX-ZF29-2026-SM-DNZYW,GTEX-ZF2S-3026-SM-4WWCH,GTEX-ZPU1-0826-SM-57WG2,GTEX-ZTPG-1426-SM-51MT3,GTEX-ZVP2-0626-SM-51MSO,GTEX-ZVT3-1626-SM-5GU66,GTEX-ZVT4-0626-SM-5E45T,GTEX-ZYT6-0626-SM-5E45V,GTEX-ZYY3-0626-SM-5NQ6W,GTEX-ZZPU-0426-SM-5GZYH
28,ENST00000374004,ENSG00000000938,0.14,0.03,1.67,0.56,0.59,1.29,0.27,1.51,...,0.98,0.86,0.56,10.85,3.22,1.52,2.45,1.76,2.06,1.63
29,ENST00000374005,ENSG00000000938,0.36,1.27,1.32,0.00,0.79,0.21,1.21,3.79,...,3.16,1.72,1.83,1.88,0.42,0.38,0.76,0.17,0.78,1.32
34,ENST00000359637,ENSG00000000971,57.68,0.00,0.00,50.96,41.76,50.15,33.72,0.00,...,0.70,0.59,0.40,26.68,35.69,31.23,39.19,75.34,52.81,0.00
35,ENST00000367429,ENSG00000000971,129.00,414.80,315.60,188.10,140.90,185.30,205.80,98.39,...,277.80,476.60,533.30,164.20,248.90,245.30,127.50,181.10,166.80,318.20
36,ENST00000466229,ENSG00000000971,28.10,31.16,27.73,61.59,48.71,46.35,86.55,6.27,...,11.40,18.87,46.23,21.18,21.82,89.37,39.80,26.14,42.09,27.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198036,ENST00000241356,ENSG00000282608,0.03,1.52,1.71,0.29,0.10,0.06,0.33,0.63,...,1.35,1.02,0.15,1.30,0.74,0.19,0.12,0.11,0.92,1.11
198166,ENST00000635200,ENSG00000282988,0.25,0.00,0.00,0.18,0.64,0.36,0.13,2.45,...,0.28,0.15,0.15,2.06,0.18,0.79,0.59,0.20,1.79,0.21
198257,ENST00000418776,ENSG00000283149,6.86,0.75,8.58,12.81,5.16,10.87,6.79,43.94,...,8.85,5.26,19.73,12.39,8.51,10.48,11.56,6.99,10.78,15.09
198298,ENST00000636204,ENSG00000283189,3.98,2.36,2.09,4.31,3.76,3.96,3.57,0.70,...,0.84,1.53,0.51,1.43,0.24,3.32,1.28,2.68,2.85,2.16


In [13]:
# gene_data_scaled = standardize_dataframe(gene_data).T
# transcript_data_scaled = standardize_dataframe(transcript_data).T
#transcript_data_scaled, gene_data_scaled, transcript_data = remove_problematic_transcripts(transcript_data_scaled, gene_data_scaled, transcript_data)

In [14]:
from alternet import postprocessing
from alternet.edge_categorization import *

In [15]:
transcript_data

,transcript_id,gene_id,GTEX-11DXY-0526-SM-5EGGQ,GTEX-11DXZ-0126-SM-5EGGY,GTEX-11EQ9-0526-SM-5A5JZ,GTEX-11GSP-0626-SM-5986T,GTEX-11NUK-1226-SM-5P9GM,GTEX-11NV4-1326-SM-5HL6V,GTEX-11OF3-0726-SM-5BC4Z,GTEX-11TT1-1726-SM-5EQLJ,...,GTEX-ZF29-2026-SM-DNZYW,GTEX-ZF2S-3026-SM-4WWCH,GTEX-ZPU1-0826-SM-57WG2,GTEX-ZTPG-1426-SM-51MT3,GTEX-ZVP2-0626-SM-51MSO,GTEX-ZVT3-1626-SM-5GU66,GTEX-ZVT4-0626-SM-5E45T,GTEX-ZYT6-0626-SM-5E45V,GTEX-ZYY3-0626-SM-5NQ6W,GTEX-ZZPU-0426-SM-5GZYH
28,ENST00000374004,ENSG00000000938,0.14,0.03,1.67,0.56,0.59,1.29,0.27,1.51,...,0.98,0.86,0.56,10.85,3.22,1.52,2.45,1.76,2.06,1.63
29,ENST00000374005,ENSG00000000938,0.36,1.27,1.32,0.00,0.79,0.21,1.21,3.79,...,3.16,1.72,1.83,1.88,0.42,0.38,0.76,0.17,0.78,1.32
34,ENST00000359637,ENSG00000000971,57.68,0.00,0.00,50.96,41.76,50.15,33.72,0.00,...,0.70,0.59,0.40,26.68,35.69,31.23,39.19,75.34,52.81,0.00
35,ENST00000367429,ENSG00000000971,129.00,414.80,315.60,188.10,140.90,185.30,205.80,98.39,...,277.80,476.60,533.30,164.20,248.90,245.30,127.50,181.10,166.80,318.20
36,ENST00000466229,ENSG00000000971,28.10,31.16,27.73,61.59,48.71,46.35,86.55,6.27,...,11.40,18.87,46.23,21.18,21.82,89.37,39.80,26.14,42.09,27.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198036,ENST00000241356,ENSG00000282608,0.03,1.52,1.71,0.29,0.10,0.06,0.33,0.63,...,1.35,1.02,0.15,1.30,0.74,0.19,0.12,0.11,0.92,1.11
198166,ENST00000635200,ENSG00000282988,0.25,0.00,0.00,0.18,0.64,0.36,0.13,2.45,...,0.28,0.15,0.15,2.06,0.18,0.79,0.59,0.20,1.79,0.21
198257,ENST00000418776,ENSG00000283149,6.86,0.75,8.58,12.81,5.16,10.87,6.79,43.94,...,8.85,5.26,19.73,12.39,8.51,10.48,11.56,6.99,10.78,15.09
198298,ENST00000636204,ENSG00000283189,3.98,2.36,2.09,4.31,3.76,3.96,3.57,0.70,...,0.84,1.53,0.51,1.43,0.24,3.32,1.28,2.68,2.85,2.16


In [16]:
canonical_grn = pd.read_csv(op.join(results_path, CONDITION, f"{CONDITION}_canonical_raw.tsv"), sep='\t')

as_source_grn = pd.read_csv(op.join(results_path,CONDITION,  f"{CONDITION}_as_aware_source_raw.tsv"), sep='\t')

fully_as_grn= pd.read_csv(op.join(results_path, CONDITION, f"{CONDITION}_fully_as_aware_raw.tsv"), sep='\t')


In [17]:
results_path

'/data/bionets/og86asub/alternet-project/alternet/results-2.0.1'

In [18]:
# Filtering Parameters
MIN_FREQUENCY = 10
IMPORTANCE_PERCENTILE = 0.7  # Keep top 30%


# Set C and Set D: PSI/Usage thresholds

DOM_MIN = 0.5
DOM_EQ_MIN = 0.7
GENE_TPM_MIN = 1.0,

# Set C specific
MIN_ISOFORMS_FOR_SPLICING = 2
TOP_M_EXPRESSED = 3



In [19]:
from alternet.alternet_class import *

In [20]:
canonical_grn

,source_gene,target_gene,frequency,mean_importance,median_importance,reg_type
0,ENSG00000001497,ENSG00000000938,1,0.037168,0.037168,TF
1,ENSG00000001497,ENSG00000000971,1,0.254878,0.254878,TF
2,ENSG00000001497,ENSG00000001036,1,21.519843,21.519843,TF
3,ENSG00000001497,ENSG00000001084,1,0.029876,0.029876,TF
4,ENSG00000001497,ENSG00000002330,1,0.232068,0.232068,TF
...,...,...,...,...,...,...
2043661,ENSG00000273841,ENSG00000281991,1,0.040733,0.040733,TF
2043662,ENSG00000273841,ENSG00000282988,1,0.380291,0.380291,TF
2043663,ENSG00000273841,ENSG00000283149,1,0.421037,0.421037,TF
2043664,ENSG00000273841,ENSG00000283189,1,0.190602,0.190602,TF


In [21]:
as_source_grn

,source_transcript,target_gene,frequency,mean_importance,median_importance,source_gene,reg_type
0,ENST00000020945,ENSG00000000938,1,0.045458,0.045458,ENSG00000019549,TF
1,ENST00000020945,ENSG00000000971,1,0.010778,0.010778,ENSG00000019549,TF
2,ENST00000020945,ENSG00000001497,1,0.177306,0.177306,ENSG00000019549,TF
3,ENST00000020945,ENSG00000001630,1,1.650970,1.650970,ENSG00000019549,TF
4,ENST00000020945,ENSG00000002330,1,0.219640,0.219640,ENSG00000019549,TF
...,...,...,...,...,...,...,...
3490382,ENST00000640075,ENSG00000281991,1,0.580847,0.580847,ENSG00000169764,TF
3490383,ENST00000640075,ENSG00000282608,1,0.071188,0.071188,ENSG00000169764,TF
3490384,ENST00000640075,ENSG00000282988,1,0.021319,0.021319,ENSG00000169764,TF
3490385,ENST00000640075,ENSG00000283149,1,0.000571,0.000571,ENSG00000169764,TF


In [22]:
alternet_obj21 = Alternet(canonical_grn, as_source_grn, fully_as_grn, transcript_data, regulator_list, tx_to_regtype , 'gene_id', 'transcript_id', min_frequency=1)

Computing set A

Set A: 973,190 rows

Category distribution:
  source_isoform_specific         426,553 ( 43.8%)
  source_gene_specific            410,899 ( 42.2%)
  source_ambiguous                 73,297 (  7.5%)
  source_equivalent                62,441 (  6.4%)
Set D diambiguation
Corrected version
Computing set D
Computing set B

Final Set B: 1,747,560 rows

Category distribution:
  target_isoform_specific         903,659 ( 51.7%)
  target_gene_specific            620,900 ( 35.5%)
  target_ambiguous                116,802 (  6.7%)
  target_equivalent               106,199 (  6.1%)

By regulator type:
  n_target_tx == 0 (gene_specific, no Net3 edges): 444,719
  n_target_tx == 1 (resolved): 1,024,325
  n_target_tx >= 2 (multi): 278,516
  target_tx_dominant non-empty: 1,302,841

Final Set C: 463,331 rows

Category distribution:
  sf_expression_associated             201,630 ( 43.5%)
  sf_ambiguous                         138,072 ( 29.8%)
  sf_splicing_supported_specific       123,626 

In [43]:
alternet_obj21.set_b

,source_transcript,source_gene,target_gene,S2_mean,S3_mean_sum,S3_mean_max,S2_median,S3_median,dominance,E2,...,max_median,reg_type,n_target_tx,target_tx_resolved,target_tx_dominant,tgt_dominance,tgt_n_isoforms,is_plausible,filter_reasons,ratio_S3_S2
1399509,ENST00000584760,ENSG00000072310,ENSG00000072310,84.306541,457.023755,63.371430,84.306541,457.023755,0.138661,True,...,457.023755,TF,13,,ENST00000581707,0.304108,14.0,True,,5.420976
423944,ENST00000476994,ENSG00000072310,ENSG00000072310,30.596072,406.457715,66.676790,30.596072,406.457715,0.164044,True,...,406.457715,TF,13,,ENST00000395757,0.304108,14.0,True,,13.284637
739834,ENST00000599385,ENSG00000105516,ENSG00000160282,55.135359,376.495908,73.206792,55.135359,376.495908,0.194442,True,...,376.495908,TF,12,,ENST00000469240,0.353846,12.0,True,,6.828574
131121,ENST00000583080,ENSG00000072310,ENSG00000072310,18.659349,371.620913,68.484035,18.659349,371.620913,0.184285,True,...,371.620913,TF,13,,ENST00000584760,0.304108,14.0,True,,19.916069
665427,ENST00000581707,ENSG00000072310,ENSG00000072310,77.736474,351.288219,58.964707,77.736474,351.288219,0.167853,True,...,351.288219,TF,13,,ENST00000478616,0.304108,14.0,True,,4.518963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81603,ENST00000326956,ENSG00000157216,ENSG00000164120,0.000000,0.187225,0.187225,0.000000,0.187225,0.999995,False,...,0.187225,TF,1,ENST00000511499,ENST00000511499,0.839809,4.0,True,,187224.770272
943266,ENST00000423266,ENSG00000166925,ENSG00000136235,0.000000,0.187225,0.187225,0.000000,0.187225,0.999995,False,...,0.187225,TF,1,ENST00000258733,ENST00000258733,0.506361,2.0,True,,187224.655189
1226540,ENST00000313285,ENSG00000125733,ENSG00000100297,0.000000,0.187224,0.187224,0.000000,0.187224,0.999995,False,...,0.187224,TF,1,ENST00000382011,ENST00000382011,1.000000,1.0,False,single_isoform_tgt;dominant_tgt_review;,187224.148764
1022349,ENST00000020945,ENSG00000019549,ENSG00000171848,0.000000,0.187224,0.187224,0.000000,0.187224,0.999995,False,...,0.187224,TF,1,ENST00000304567,ENST00000304567,0.999999,1.0,False,single_isoform_tgt;dominant_tgt_review;,187223.996630


In [24]:
set_b = alternet_obj21.set_b.copy()
net3 = alternet_obj21.as_full.copy()

In [25]:
t2 = net3.sort_values('median_importance', ascending=False)[0:500]

In [26]:
for g in t2['target_transcript']:
    print(g)

ENST00000489682
ENST00000307145
ENST00000558673
ENST00000480865
ENST00000599385
ENST00000376112
ENST00000532091
ENST00000008938
ENST00000555686
ENST00000488212
ENST00000435381
ENST00000556324
ENST00000405333
ENST00000375650
ENST00000461414
ENST00000335681
ENST00000471741
ENST00000375651
ENST00000577640
ENST00000237530
ENST00000527983
ENST00000382011
ENST00000409409
ENST00000469240
ENST00000535788
ENST00000554617
ENST00000565143
ENST00000578469
ENST00000305786
ENST00000471651
ENST00000602786
ENST00000620739
ENST00000493708
ENST00000433408
ENST00000554617
ENST00000470384
ENST00000164227
ENST00000584760
ENST00000581707
ENST00000395756
ENST00000487982
ENST00000377532
ENST00000300098
ENST00000269305
ENST00000369085
ENST00000366983
ENST00000469435
ENST00000531063
ENST00000395757
ENST00000460806
ENST00000496823
ENST00000257549
ENST00000535434
ENST00000556140
ENST00000466898
ENST00000403534
ENST00000375650
ENST00000306279
ENST00000558674
ENST00000492343
ENST00000518206
ENST00000301634
ENST0000

In [46]:
import plotly.graph_objects as go

# 1. Define the Labels (Nodes)
# Index 0: Source, Index 1: Group A, Index 2: Group B
labels = ["Initial Messages (10)", "Group A (10 Recipients)", "Group B (10 Recipients)"]

# 2. Define the Links (The Flows)
# source: where the flow starts (using node index)
# target: where the flow ends
# value: the thickness of the line
fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 20,
      thickness = 30,
      line = dict(color = "black", width = 0.5),
      label = labels,
      color = "navy"
    ),
    link = dict(
      source = [0, 0],   # Both start at 'Initial Messages'
      target = [1, 2],   # One goes to Group A, one to Group B
      value = [10, 10],  # 10 units each
      color = "rgba(0, 0, 128, 0.4)" # Semi-transparent navy
  ))])

fig.update_layout(title_text="Message Duplication Flow", font_size=12)
fig.write_html("my_sankey.html")

In [27]:
def score_targets(edges, target_col, importance_col='median_importance'):
    """
    Compute target scores as weighted in-degree.
    score(target) = sum of importance for all edges targeting it
    """
    scores = edges.groupby(target_col)[importance_col].sum()
    return scores.sort_values(ascending=False)


In [28]:
def project_tx_to_gene(tx_scores, tx2gene, method='max'):
    """
    Project transcript-level scores to gene-level.
    
    Parameters:
    - tx_scores: pd.Series mapping transcript_id -> score
    - tx2gene: dict mapping transcript_id -> gene_id
    - method: 'max' (default) or 'sum'
    
    Returns:
    - gene_scores: pd.Series mapping gene_id -> score
    - rep_tx: dict mapping gene_id -> representative transcript
    """
    df = pd.DataFrame({
        'transcript_id': tx_scores.index,
        'score': tx_scores.values
    })
    df['gene_id'] = df['transcript_id'].map(tx2gene)
    df = df.dropna(subset=['gene_id'])
    
    if method == 'max':
        idx = df.groupby('gene_id')['score'].idxmax()
        result = df.loc[idx].set_index('gene_id')
        gene_scores = result['score'].sort_values(ascending=False)
        rep_tx = result['transcript_id'].to_dict()
    elif method == 'sum':
        gene_scores = df.groupby('gene_id')['score'].sum().sort_values(ascending=False)
        rep_tx = {}
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return gene_scores, rep_tx


In [29]:
def build_target_list(edges, target_col, importance_col='median_importance',
                      target_type='gene', tx2gene=None, gene2symbol=None,
                      K=500, projection_method='max'):
    """
    Build a ranked target list from an edge set.
    
    Parameters:
    - edges: DataFrame with edges
    - target_col: column containing target IDs
    - importance_col: column containing importance scores
    - target_type: 'gene' or 'transcript'
    - tx2gene: transcript to gene mapping (required if target_type='transcript')
    - gene2symbol: gene ID to symbol mapping
    - K: number of top genes to return
    - projection_method: 'max' or 'sum' (for transcript targets)
    
    Returns:
    - top_df: DataFrame with ranked targets
    - gene_symbols: list of gene symbols for g:Profiler
    - metadata: dict with scoring info
    """
    if len(edges) == 0:
        return pd.DataFrame(), [], {'n_edges': 0}
    
    # Score targets
    target_scores = score_targets(edges, target_col, importance_col)
    
    # Project if transcript targets
    if target_type == 'transcript':
        if tx2gene is None:
            raise ValueError("tx2gene mapping required for transcript targets")
        gene_scores, rep_tx = project_tx_to_gene(target_scores, tx2gene, method=projection_method)
    else:
        gene_scores = target_scores
        rep_tx = {}
    
    # Get top K
    top_genes = gene_scores.head(K)
    
    # Build result DataFrame
    top_df = pd.DataFrame({
        'rank': range(1, len(top_genes) + 1),
        'gene_id': top_genes.index,
        'score': top_genes.values
    })
    
    if gene2symbol is not None:
        top_df['gene_symbol'] = top_df['gene_id'].map(gene2symbol)
        top_df = top_df.dropna(subset=['gene_symbol'])
        gene_symbols = top_df['gene_symbol'].tolist()
    else:
        gene_symbols = top_df['gene_id'].tolist()
    
    if len(rep_tx) > 0:
        top_df['rep_transcript'] = top_df['gene_id'].map(rep_tx)
    
    metadata = {
        'n_edges': len(edges),
        'n_targets': len(target_scores),
        'n_genes': len(gene_scores),
        'n_top_symbols': len(gene_symbols),
        'projection_method': projection_method if target_type == 'transcript' else 'none'
    }
    
    return top_df, gene_symbols, metadata

In [30]:

tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2symbol = dict(zip(biomart['Gene stable ID'], biomart['Gene name']))
symbol2gene = dict(zip(biomart['Gene name'], biomart['Gene stable ID']))

In [31]:
TOP_K = 500
PROJECTION_METHOD = 'max'

In [32]:

# Filter Set B for target_isoform_unique edges
l3_table2 = set_b[set_b['target_category'] == 'target_isoform_specific'].copy()

# Get corresponding Net3 edges for transcript-level scoring
# Set B is at (regulator_tx, target_gene) level
# We need to link to Net3 which has (source_transcript, target_transcript)

# Strategy: Filter Net3 to TF-like edges and get transcript targets
net3_tf_like = net3[net3['reg_type'].isin(['TF', 'TF_SF'])].copy()

# Create lookup from Set B
l3_pairs = set(zip(l3_table2['source_transcript'], l3_table2['target_gene']))

# Add target_gene to Net3 for matching
net3_tf_like['target_gene'] = net3_tf_like['target_transcript'].map(tx2gene)
net3_tf_like['pair'] = list(zip(net3_tf_like['source_transcript'], net3_tf_like['target_gene']))

# Filter to edges matching Set B target_isoform_unique
l3_edges = net3_tf_like[net3_tf_like['pair'].isin(l3_pairs)].copy()

# Build target list - targets are transcripts, project to genes
l3_top, l3_symbols, l3_meta = build_target_list(
    edges=l3_edges,
    target_col='target_transcript',
    importance_col='mean_importance',
    target_type='transcript',
    tx2gene=tx2gene,
    gene2symbol=gene2symbol,
    K=TOP_K,
    projection_method=PROJECTION_METHOD
)



In [33]:
for g in l3_top.gene_symbol:
    print(g)

ACAA1
ELF3
GLYCTK
GRHPR
SREBF1
ACOX2
FOS
ALDOA
MLXIPL
SLC5A6
SLC22A7
SGK1
PAH
LIME1
PKM
ASL
FTCD
UGP2
SLC30A10
IFI44
NR1I3
IRF7
BCL6
DCXR
BDH1
PCK2
NR4A1
WWC1
AASS
GLYAT
LIMK2
ELOVL6
FASN
RPLP0
HAAO
ACY1
RALGDS
GALT
ZFAND5
UPB1
FAHD2A
AKR1A1
STAT3
DHRS1
SLC27A5
ENO1
ALDH2
C1R
P4HB
SLC6A1
NNMT
EIF4A1
SLC2A4RG
CTH
SLC13A5
CYP27A1
ATP5IF1
HGD
TKFC
CRP
SOD2
MSTO1
ATF3
HSD17B6
ABCA6
SLC46A3
SHMT2
SERPINA3
FMO3
FGG
CAT
MOGAT2
MAPKAPK2
GABARAPL1
BPHL
ADH4
NR1I2
THBS1
TIMP1
QPRT
HSPE1
FAXDC2
SDS
HAGH
EFNA1
ABLIM3
GPR108
CASP4
KLF6
ASPDH
SLC39A5
ACSM5
CLU
ETFA
GCGR
IQGAP2
MKNK2
ADH1B
FXYD1
GPAM
CYP2C8
GAPDH
PGK1
RAB26
EPHX2
ADM
SHROOM1
TMT1A
IL1RN
ACSM2A
CRELD2
CES4A
CCDC152
SEC14L2
ASS1
AGXT
RTKN
SLCO2B1
CMBL
COQ10A
PNRC1
PDIA3
GDI1
SERPINA1
TPI1
FN1
AR
ST6GAL1
PLG
ACAA2
LCAT
CLTC
ECHDC2
GUK1
ENTPD5
NDRG1
FNDC4
GLS2
CFB
C1S
DAO
SERPINA5
BCL3
AEBP1
ADH1A
RARRES2
SDHA
LRRC75B
HMGCS2
PER1
RHOQ
MAT1A
SYVN1
TGFB3
NUCB2
RNF128
FLNA
WEE1
GLYATL1
PFKFB1
PCYT2
ART4
LDHA
FBLN5
VEGFA
MAP2K3
GSTK1
PDIA4
E

In [34]:
gene_dominance, gene_n_isoforms, tx_expression_share = compute_dominance_metrics(alternet_obj21.transcript_data, alternet_obj21.sample_cols, min_tpm=0.1, epsilon = 1e-6)

In [35]:
set_a = filter_set_a(alternet_obj21.set_a, gene_dominance, gene_n_isoforms,
                  dom_threshold=0.9, fc_eq_max=2.0, eps=1e-6)

In [36]:
set_a[(set_a.is_plausible) & (set_a.source_category == 'source_equivalent') ]

,edge_key,source_gene,target_gene,best_tx,S1_mean,S2_mean,S1_median,S2_median,E1,E2,ratio,source_category,max_median,reg_dominance,reg_n_isoforms,is_plausible,filter_reasons,ratio_S2_S1
330429,ENSG00000044574_ENSG00000145050,ENSG00000044574,ENSG00000145050,ENST00000324460,77.818551,72.954213,77.818551,72.954213,True,True,0.937491,source_equivalent,77.818551,1.000000,1,True,,0.937491
681212,ENSG00000102878_ENSG00000265690,ENSG00000102878,ENSG00000265690,ENST00000522870,77.558601,71.507112,77.558601,71.507112,True,True,0.921975,source_equivalent,77.558601,0.221341,7,True,,0.921975
815792,ENSG00000012223_ENSG00000008438,ENSG00000012223,ENSG00000008438,ENST00000426532,70.828464,77.234951,70.828464,77.234951,True,True,1.090451,source_equivalent,77.234951,0.999999,1,True,,1.090451
765034,ENSG00000066336_ENSG00000115956,ENSG00000066336,ENSG00000115956,ENST00000378538,74.323480,66.483798,74.323480,66.483798,True,True,0.894519,source_equivalent,74.323480,0.733109,2,True,,0.894519
227597,ENSG00000101412_ENSG00000167900,ENSG00000101412,ENSG00000167900,ENST00000343380,62.371655,72.587169,62.371655,72.587169,True,True,1.163785,source_equivalent,72.587169,0.999999,1,True,,1.163785
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187921,ENSG00000090661_ENSG00000234719,ENSG00000090661,ENSG00000234719,ENST00000558302,0.410020,0.353605,0.410020,0.353605,True,True,0.862409,source_equivalent,0.410020,0.357100,5,True,,0.862409
453328,ENSG00000169136_ENSG00000175130,ENSG00000169136,ENSG00000175130,ENST00000600336,0.410002,0.376494,0.410002,0.376494,True,True,0.918271,source_equivalent,0.410002,0.898471,5,True,,0.918271
404875,ENSG00000163110_ENSG00000127472,ENSG00000163110,ENSG00000127472,ENST00000437932,0.409988,0.367403,0.409988,0.367403,True,True,0.896129,source_equivalent,0.409988,0.519050,2,True,,0.896129
235486,ENSG00000075785_ENSG00000205362,ENSG00000075785,ENSG00000205362,ENST00000493186,0.409985,0.397512,0.409985,0.397512,True,True,0.969575,source_equivalent,0.409985,0.999999,1,True,,0.969575


In [37]:
alternet_obj21.set_a

,edge_key,source_gene,target_gene,best_tx,S1_mean,S2_mean,S1_median,S2_median,E1,E2,ratio,source_category,max_median,reg_dominance,reg_n_isoforms,is_plausible,filter_reasons,ratio_S2_S1
626759,ENSG00000079616_ENSG00000079616,ENSG00000079616,ENSG00000079616,ENST00000568312,0.0,222.180328,0.0,222.180328,False,True,inf,source_isoform_specific,222.180328,1.000000,1,False,single_isoform_reg;,2.221803e+08
128625,ENSG00000064961_ENSG00000064961,ENSG00000064961,ENSG00000064961,ENST00000470356,0.0,185.119161,0.0,185.119161,False,True,inf,source_isoform_specific,185.119161,0.999999,1,False,single_isoform_reg;,1.851192e+08
232719,ENSG00000170485_ENSG00000170485,ENSG00000170485,ENSG00000170485,ENST00000335681,0.0,179.174978,0.0,179.174978,False,True,inf,source_isoform_specific,179.174978,0.536484,3,True,,1.791750e+08
616052,ENSG00000250571_ENSG00000250571,ENSG00000250571,ENSG00000250571,ENST00000340042,0.0,174.900887,0.0,174.900887,False,True,inf,source_isoform_specific,174.900887,1.000000,1,False,single_isoform_reg;,1.749009e+08
50242,ENSG00000155090_ENSG00000155090,ENSG00000155090,ENSG00000155090,ENST00000285407,0.0,172.693116,0.0,172.693116,False,True,inf,source_isoform_specific,172.693116,1.000000,1,False,single_isoform_reg;,1.726931e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900568,ENSG00000115524_ENSG00000138303,ENSG00000115524,ENSG00000138303,ENST00000487698,0.0,0.214080,0.0,0.214080,False,True,inf,source_isoform_specific,0.214080,0.689686,4,True,,2.140804e+05
252287,ENSG00000168066_ENSG00000250479,ENSG00000168066,ENSG00000250479,ENST00000472725,0.0,0.214080,0.0,0.214080,False,True,inf,source_isoform_specific,0.214080,0.633406,4,True,,2.140799e+05
719936,ENSG00000102103_ENSG00000175482,ENSG00000102103,ENSG00000175482,ENST00000376563,0.0,0.214078,0.0,0.214078,False,True,inf,source_isoform_specific,0.214078,0.607391,4,True,,2.140777e+05
493380,ENSG00000096746_ENSG00000132541,ENSG00000096746,ENSG00000132541,ENST00000491200,0.0,0.214078,0.0,0.214078,False,True,inf,source_isoform_specific,0.214078,0.375859,5,True,,2.140775e+05


In [38]:
alternet_obj21.set_b

,source_transcript,source_gene,target_gene,S2_mean,S3_mean_sum,S3_mean_max,S2_median,S3_median,dominance,E2,...,max_median,reg_type,n_target_tx,target_tx_resolved,target_tx_dominant,tgt_dominance,tgt_n_isoforms,is_plausible,filter_reasons,ratio_S3_S2
1399509,ENST00000584760,ENSG00000072310,ENSG00000072310,84.306541,457.023755,63.371430,84.306541,457.023755,0.138661,True,...,457.023755,TF,13,,ENST00000581707,0.304108,14.0,True,,5.420976
423944,ENST00000476994,ENSG00000072310,ENSG00000072310,30.596072,406.457715,66.676790,30.596072,406.457715,0.164044,True,...,406.457715,TF,13,,ENST00000395757,0.304108,14.0,True,,13.284637
739834,ENST00000599385,ENSG00000105516,ENSG00000160282,55.135359,376.495908,73.206792,55.135359,376.495908,0.194442,True,...,376.495908,TF,12,,ENST00000469240,0.353846,12.0,True,,6.828574
131121,ENST00000583080,ENSG00000072310,ENSG00000072310,18.659349,371.620913,68.484035,18.659349,371.620913,0.184285,True,...,371.620913,TF,13,,ENST00000584760,0.304108,14.0,True,,19.916069
665427,ENST00000581707,ENSG00000072310,ENSG00000072310,77.736474,351.288219,58.964707,77.736474,351.288219,0.167853,True,...,351.288219,TF,13,,ENST00000478616,0.304108,14.0,True,,4.518963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81603,ENST00000326956,ENSG00000157216,ENSG00000164120,0.000000,0.187225,0.187225,0.000000,0.187225,0.999995,False,...,0.187225,TF,1,ENST00000511499,ENST00000511499,0.839809,4.0,True,,187224.770272
943266,ENST00000423266,ENSG00000166925,ENSG00000136235,0.000000,0.187225,0.187225,0.000000,0.187225,0.999995,False,...,0.187225,TF,1,ENST00000258733,ENST00000258733,0.506361,2.0,True,,187224.655189
1226540,ENST00000313285,ENSG00000125733,ENSG00000100297,0.000000,0.187224,0.187224,0.000000,0.187224,0.999995,False,...,0.187224,TF,1,ENST00000382011,ENST00000382011,1.000000,1.0,False,single_isoform_tgt;dominant_tgt_review;,187224.148764
1022349,ENST00000020945,ENSG00000019549,ENSG00000171848,0.000000,0.187224,0.187224,0.000000,0.187224,0.999995,False,...,0.187224,TF,1,ENST00000304567,ENST00000304567,0.999999,1.0,False,single_isoform_tgt;dominant_tgt_review;,187223.996630


In [39]:
set_b = filter_set_b(alternet_obj21.set_b, gene_dominance, gene_n_isoforms,
                  dom_threshold=0.9, fc_eq_max=2.0, eps=1e-6)

In [40]:
set_b

,source_transcript,source_gene,target_gene,S2_mean,S3_mean_sum,S3_mean_max,S2_median,S3_median,dominance,E2,...,max_median,reg_type,n_target_tx,target_tx_resolved,target_tx_dominant,tgt_dominance,tgt_n_isoforms,is_plausible,filter_reasons,ratio_S3_S2
1399509,ENST00000584760,ENSG00000072310,ENSG00000072310,84.306541,457.023755,63.371430,84.306541,457.023755,0.138661,True,...,457.023755,TF,13,,ENST00000581707,0.304108,14.0,True,,5.420976
423944,ENST00000476994,ENSG00000072310,ENSG00000072310,30.596072,406.457715,66.676790,30.596072,406.457715,0.164044,True,...,406.457715,TF,13,,ENST00000395757,0.304108,14.0,True,,13.284637
739834,ENST00000599385,ENSG00000105516,ENSG00000160282,55.135359,376.495908,73.206792,55.135359,376.495908,0.194442,True,...,376.495908,TF,12,,ENST00000469240,0.353846,12.0,True,,6.828574
131121,ENST00000583080,ENSG00000072310,ENSG00000072310,18.659349,371.620913,68.484035,18.659349,371.620913,0.184285,True,...,371.620913,TF,13,,ENST00000584760,0.304108,14.0,True,,19.916069
665427,ENST00000581707,ENSG00000072310,ENSG00000072310,77.736474,351.288219,58.964707,77.736474,351.288219,0.167853,True,...,351.288219,TF,13,,ENST00000478616,0.304108,14.0,True,,4.518963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81603,ENST00000326956,ENSG00000157216,ENSG00000164120,0.000000,0.187225,0.187225,0.000000,0.187225,0.999995,False,...,0.187225,TF,1,ENST00000511499,ENST00000511499,0.839809,4.0,True,,187224.770272
943266,ENST00000423266,ENSG00000166925,ENSG00000136235,0.000000,0.187225,0.187225,0.000000,0.187225,0.999995,False,...,0.187225,TF,1,ENST00000258733,ENST00000258733,0.506361,2.0,True,,187224.655189
1226540,ENST00000313285,ENSG00000125733,ENSG00000100297,0.000000,0.187224,0.187224,0.000000,0.187224,0.999995,False,...,0.187224,TF,1,ENST00000382011,ENST00000382011,1.000000,1.0,False,single_isoform_tgt;dominant_tgt_review;,187224.148764
1022349,ENST00000020945,ENSG00000019549,ENSG00000171848,0.000000,0.187224,0.187224,0.000000,0.187224,0.999995,False,...,0.187224,TF,1,ENST00000304567,ENST00000304567,0.999999,1.0,False,single_isoform_tgt;dominant_tgt_review;,187223.996630
